In [ ]:
from pyfiles.preamble import *
enable_autoreload()

from pyfiles.create_scenario import (
    BASE_PARAMS, BASE_CASE_PARAMS, SHOCK_CASE_PARAMS,
    create_scenarios,
)
create_scenarios()

# Cost minimisation (5_cost metric)

Finds the capacity vector $(PP2, ELT, H_2)$ that minimises the **5_cost_decomposition metric** — not EnergyPLAN's total annual cost. The approach starts from TAC (which covers ~95% of the metric automatically) and applies two targeted corrections:

1. **Subtract CSHP investment** — `Indust. CHP Heat` double-counts the nuclear DH investment already captured in the nuclear unit cost.
2. **Add external system costs** — grid integration cost at 145 DKK/MWh for VRE production and 10 DKK/MWh for nuclear, consistent with `5_cost_decomposition.ipynb`.

Because more ELT reduces VRE curtailment, solver.py (which minimises TAC) over-invests in ELT: absorbed curtailed VRE looks free in TAC but carries the 145 DKK/MWh system cost rate here. The bias is **against nuclear** (solver.py makes nuclear's system-cost advantage look smaller than it is).

The optimisation problem:
\begin{gather*}
    \arg\min_{PP2,\,ELT,\,H_2} \left[
        TAC - I_{\text{CSHP}} + Q_{\text{VRE}} \cdot \frac{145}{7.45} + Q_{\text{KK}} \cdot \frac{10}{7.45}
    \right]
    \quad \text{s.t.} \quad M \leq 0.8\,\text{TWh}
\end{gather*}
where $I_{\text{CSHP}}$ is the annualised investment + O\&M for `Indust. CHP Heat`, $Q$ is annual production in TWh, and rates are in DKK/MWh converted at 7.45 DKK/EUR.

In [ ]:
import ctypes
import pyfiles.solver as solver
ctypes.windll.kernel32.SetThreadExecutionState(0x80000001 | 0x00000002)

In [ ]:
solver.IMPORT_LIMIT_TWH = 0.8

solver.configure('base', BASE_PARAMS, BASE_CASE_PARAMS, SHOCK_CASE_PARAMS)
out_base_08 = solver.run(max_evaluations=500)

In [ ]:
solver.IMPORT_LIMIT_TWH = 0.8

solver.configure('shock', BASE_PARAMS, BASE_CASE_PARAMS, SHOCK_CASE_PARAMS)
out_shock_08 = solver.run(max_evaluations=500)

In [ ]:
ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)

**Results log** — update after each run:
